In [1]:
import sys

from tabpfn import TabPFNClassifier
import importlib
from pathlib import Path
from datetime import datetime

# walk up from the current folder until we find the one containing utils/
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "utils").is_dir():
        sys.path.insert(0, str(p))
        print("Repo root:", p)
        break
else:
    raise RuntimeError(f"utils/ not found from {here}")

import joblib
import pandas as pd


from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from utils.training_automatisation import log_run, run_experiment
from utils.config import DATA_DIR
from utils.config import EXPORT_DIR




Repo root: /home/paul/pers/kaggle.playground.11.26


/home/paul/anaconda3/envs/kaggle_UMUD/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


# Data files
data_dir = Path(DATA_DIR)
train_file = data_dir / "df_encoded_train_hot_ones.csv"  # Set to your encoded training file
test_file = data_dir / "df_encoded_test_hot_ones.csv"  # Set to your encoded test file

target = "is_fraud"
id_column = "id"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
feature_columns = [col for col in train_df.columns if col != target and col != id_column]


X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]



In [3]:
num_cols = [c for c in X_train.select_dtypes(include="number").columns
            if X_train[c].nunique() > 2]

scaler = ColumnTransformer([("scale", StandardScaler(), num_cols)],
                           remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#model to train 

MODELS = {
    "logistic_regression_basic": {
        "active": False,
        "type": "linear",
        "model": Pipeline([
            ("preprocess", scaler),
            ("clf", LogisticRegression(class_weight="balanced",
                                       max_iter=1000, random_state=42)),
        ]),
    },
    "random_forest": {
        "active": False,
        "type": "tree",
        "model": Pipeline([
            ("preprocess", "passthrough"),
            ("clf", RandomForestClassifier(class_weight="balanced",
                                           n_jobs=1, random_state=42)),
        ]),
        "param_grid": {
            "clf__n_estimators": [500],
            "clf__max_depth": [None],
            "clf__min_samples_leaf": [1,5,10],
            "clf__min_samples_split": [2,5,10],
        },
    },
        "xgboost": {
        "active": False,
        "type": "boosting",
        "model": Pipeline([
            ("preprocess", "passthrough"),
            ("clf", XGBClassifier(
                objective="binary:logistic",
                eval_metric="auc",
                scale_pos_weight=(
                    (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                ),
                n_estimators=500,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                n_jobs=1,
                random_state=42,
                tree_method="hist",
device="cuda",
            )),
        ]),
        "param_grid": {
    "clf__max_depth": [3],
    "clf__min_child_weight": [5],
    "clf__subsample": [0.7, 0.85, 1.0],
    "clf__colsample_bytree": [0.7, 0.85, 1.0],
}
    },
    "lightgbm": {
        "active":  False,
        "type": "boosting",
        "model": Pipeline([
            ("preprocess", "passthrough"),
            ("clf", LGBMClassifier(
                objective="binary",
                class_weight="balanced",
                n_estimators=500,
                learning_rate=0.05,
                n_jobs=1,
                random_state=42,
                verbosity=-1,
                device_type="gpu",
            )),
        ]),
        "param_grid": {
    "clf__num_leaves": [15],
    "clf__min_child_samples": [ 50],
        "clf__subsample": [0.7, 0.85, 1.0],
    "clf__colsample_bytree": [0.7, 0.85, 1.0],
}
    },
    "catboost": {
    "active": False,
    "type": "boosting",
    "model": Pipeline([
        ("preprocess", "passthrough"),
        ("clf", CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            auto_class_weights="Balanced",
            iterations=800,
            learning_rate=0.05,
            task_type="GPU",
            thread_count=1,
            random_seed=42,
            verbose=False,
        )),
    ]),
    "param_grid": {
        "clf__depth": [5, 7],
        "clf__l2_leaf_reg": [3, 7],
    },
    
},
    "tabpfn_26": {
    "active": True,
    "type": "foundation",
    "tabpfn": {
        "max_training_rows": 10_000,
        "max_features": 500,
    },
    "model": Pipeline([
        ("preprocess", "passthrough"),
        ("clf", TabPFNClassifier(
            device="cuda",
            n_estimators=4,
            random_state=42,
            ignore_pretraining_limits=True,
        )),
    ]),
},
}


In [4]:
models_dir = Path(EXPORT_DIR) / "models_weights"

to_run = {n: s for n, s in MODELS.items() if s.get("active", True)}

for name, spec in to_run.items():
    run_experiment(name, spec, X_train, y_train, X_test, test_df, id_column, models_dir, data_dir, cv=cv)

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/paul/anaconda3/envs/kaggle_UMUD/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/paul/anaconda3/envs/kaggle_UMUD/lib/python3.14/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/paul/anaconda3/envs/kaggle_UMUD/lib/python3.14/site-packages/sklearn/pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^